In [ ]:
!git clone https://github.com/facebookresearch/sam2.git

In [ ]:
cd sam2

In [ ]:
!pip -q install -e .

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

In [ ]:
import os, sys
os.environ["SAM2_REPO_ROOT"] = "/kaggle/working/sam2"
os.environ["PYTHONPATH"] = f"{os.environ['SAM2_REPO_ROOT']}:{os.environ.get('PYTHONPATH','')}"
sys.path.insert(0, os.environ["SAM2_REPO_ROOT"])


In [ ]:
from sam2.build_sam import build_sam2_video_predictor

sam2_checkpoint = "/kaggle/input/sam2-1-hiera-large/pytorch/default/1/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)

In [ ]:
def show_mask(mask, ax, obj_id=None, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        cmap = plt.get_cmap("tab10")
        cmap_idx = 0 if obj_id is None else obj_id
        color = np.array([*cmap(cmap_idx)[:3], 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_points(coords, labels, ax, marker_size=200):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)


def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))

In [ ]:
video_dir = "/kaggle/input/v155-dataset"

# scan all the JPEG frame names in this directory
frame_names = [
    p for p in os.listdir(video_dir)
    if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
]
frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))

# take a look of the video frame with the object
frame_idx = 100
plt.figure(figsize=(9, 6))
plt.title(f"frame {frame_idx}")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[frame_idx])))

In [ ]:
inference_state = predictor.init_state(video_path=video_dir)

In [ ]:
ann_frame_idx = 100  # the frame index we interact with
ann_obj_id = 1  # give a unique id to each object we interact with (it can be any integers)

# Add a positive click at (x, y)
points = np.array([[70, 260], [100, 250],[140,240],[180, 230]
                   ], dtype=np.float32)


# for labels, `1` means positive click and `0` means negative click
labels = np.array([1,1,1,1], np.int32)
_, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)

# show the results on the current (interacted) frame
plt.figure(figsize=(9, 6))
plt.title(f"frame {ann_frame_idx}")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[ann_frame_idx])))
show_points(points, labels, plt.gca())
show_mask((out_mask_logits[0] > 0.0).cpu().numpy(), plt.gca(), obj_id=out_obj_ids[0])

In [ ]:
# helpers
def update_segments(video_segments, video_scores, frame_idx, obj_id, logits):
    # pick the result with higher mean logit (confidence-ish)
    score = float(logits.mean().cpu())
    prev = video_scores[frame_idx].get(obj_id, -1e9)
    if score > prev:
        video_scores[frame_idx][obj_id] = score
        mask = (logits > 0.0).cpu().numpy()
        if mask.any():  # keep only non-empty
            video_segments[frame_idx][obj_id] = mask

# storage
num_frames = len(frame_names)
video_segments = {i: {} for i in range(num_frames)}
video_scores   = {i: {} for i in range(num_frames)}  # for merging

# pass 1: forward (covers prompt frame -> end)
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
    inference_state, reverse=False
):
    for i, out_obj_id in enumerate(out_obj_ids):
        update_segments(video_segments, video_scores, out_frame_idx, out_obj_id, out_mask_logits[i])

# pass 2: backward (covers prompt frame -> beginning)
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
    inference_state, reverse=True
):
    for i, out_obj_id in enumerate(out_obj_ids):
        update_segments(video_segments, video_scores, out_frame_idx, out_obj_id, out_mask_logits[i])

# render
vis_frame_stride = 20
plt.close("all")
for out_frame_idx in range(0, num_frames, vis_frame_stride):
    plt.figure(figsize=(6, 4))
    plt.title(f"frame {out_frame_idx}")
    plt.imshow(Image.open(os.path.join(video_dir, frame_names[out_frame_idx])))

    for out_obj_id, out_mask in video_segments[out_frame_idx].items():
        show_mask(out_mask, plt.gca(), obj_id=out_obj_id)

    plt.axis("off")
    plt.show()


In [ ]:
import os, numpy as np
from PIL import Image

video_dir = video_dir
H, W = Image.open(os.path.join(video_dir, frame_names[0])).size[1], Image.open(os.path.join(video_dir, frame_names[0])).size[0]  # (height, width)

def to_hw_uint8(mask, H, W):
    """Convert various mask shapes to (H, W) uint8 in {0,1}."""
    arr = np.asarray(mask)

    # squeeze singleton dims
    arr = np.squeeze(arr)

    # reshape if flat
    if arr.ndim == 1 and arr.size == H * W:
        arr = arr.reshape(H, W)

    # handle weird (1,1,H*W) that became (H*W,) after squeeze earlier
    if arr.ndim == 3 and arr.shape[-1] == H * W and arr.shape[0] == 1 and arr.shape[1] == 1:
        arr = arr.reshape(H, W)

    # final safety: if channels-last
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    # channels-first
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]

    assert arr.ndim == 2, f"Mask not 2D after normalization, got shape {arr.shape}"

    # binarize (>0) and cast
    arr = (arr > 0).astype(np.uint8)
    return arr

# save combined per-frame masks (black/white)
save_dir = "/kaggle/working/masks"
os.makedirs(save_dir, exist_ok=True)

saved = 0
for frame_idx, obj_dict in video_segments.items():
    if not obj_dict:
        continue
    combined = np.zeros((H, W), dtype=np.uint8)
    for mask in obj_dict.values():
        m = to_hw_uint8(mask, H, W)
        combined |= m
    Image.fromarray(combined * 255, mode="L").save(os.path.join(save_dir, f"frame_{frame_idx:04d}.png"))
    saved += 1

print(f"Saved {saved} combined masks to {save_dir}")
